 https://jse.amstat.org/v19n3/decock/DataDocumentation.txt
 

In [1]:
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

# Encoding Categorical Columns

Categorical columns must be converted to numbers before being passed to any model. The choice of encoding strategy depends on three factors:

- whether the categories have a natural order,
- how many unique values the column has (cardinality),
- and how frequently each category appears in the data.

```python
df.select_dtypes(include="object").columns
df[col].nunique()        # number of unique categories in a column, CARDINALITY
df[col].value_counts()   # frequency of each category
```

---

# Decision Grounds Based

Which encoding to use depends on cardinality and the model type. Remove ordinal columns first and handle them separately, then apply the decision below to the remaining unordered categorical columns.

**Tree-based models (XGBoost, LightGBM, Random Forest)**

Trees make binary splits: "is this feature greater than threshold X?" They do not care about the distance or order between integer values, only whether a split separates the data well.

| Cardinality | Recommended encoding |
|---|---|
| Ordered (quality, condition) | Ordinal encoding with explicit order |
| Low, unordered (< 10) | Label encoding (`pd.factorize`) is sufficient |
| High, unordered (10+) | Smoothed target encoding or CatBoost encoding |

**Linear models / Neural networks**

These models treat input values as continuous numbers where magnitude and distance matter. An arbitrary label encoding like `BrDale=0, Blmngtn=1, BrkSide=2` implies BrkSide is twice Blmngtn, which is meaningless and misleads the model.

| Cardinality | Recommended encoding |
|---|---|
| Ordered (quality, condition) | Ordinal encoding with explicit order |
| Low, unordered (< 10) | One-hot encoding |
| Medium, unordered (10-50) | Target encoding with k-fold or smoothing |
| High, unordered (50+) | Target encoding or `nn.Embedding` |
| Too many columns after one-hot | Binary encoding as a compromise |

---
# IMPORTANT: Handling Unknown Categories at Inference Time

A category present in the test set but absent from the training set will break any encoding that relies on a lookup table. Always plan for this.

| Encoding          | Strategy for unknowns                                        |     |
| ----------------- | ------------------------------------------------------------ | --- |
| One-hot           | `handle_unknown="ignore"` in sklearn (produces all-zero row) |     |
| Ordinal           | `handle_unknown="use_encoded_value", unknown_value=-1`       |     |
| Label             | `OrdinalEncoder` with `unknown_value=-1`                     |     |
| Target / Smoothed | Fill with global target mean                                 |     |
| Frequency         | Fill with 0                                                  |     |
| Rare grouping     | Map to `"Other"` before encoding                             |     |

# IMPORTANT: Fit-Tranform expecting 2D Array

Fit transform function used down there expects an 2D array, as normally its used for the entire dataset in `(x_samples, y_features)` shape, like:

```
LotArea   GrLivArea   YearBuilt
ev_1      8450      1710        2003        → 1 sample, 3 feature
ev_2      9600      1262        1976
ev_3      11250     1786        2001
...
ev_1460   9000      1831        1999

shape: (1460, 3)   →   1460 sample, 3 feature
```

So while using for 1D array with just 1 feature with x_samples, it must be put in a 2D array like `[[data]]`.


# TRICK: Setoutput to recieve pandas dataframe directly instead of np array to concat:
`oe.set_output(transform="pandas")` setting output of the transformer/encoder like that makes pd.concat command easier



---
---

## Possible Strategies

### Standard Strategies

**1. Ordinal Encoding**

Used when categories have a meaningful, rankable order. Never use `LabelEncoder` for ordinal columns without specifying the order, as it assigns integers alphabetically and may not match the true order.

```python
from sklearn.preprocessing import OrdinalEncoder

quality_order = [["Po", "Fa", "TA", "Gd", "Ex"]]

encoder = OrdinalEncoder(
    categories=quality_order,    handle_unknown="use_encoded_value",    unknown_value=-1        # unseen categories at inference get -1)
df["Exter Qual_enc"] = encoder.fit_transform(df[["Exter Qual"]])
```



---

**2. One-Hot Encoding**

Use when categories have no natural order and cardinality is low (roughly under 10 unique values). Each unique category becomes a new binary column.

AVOID ONE-HOT ENCODING WHEN CARDINALITY IS ABOVE 10 TO PREVENT DIMENSIONALITY EXPLOSION.

*What does drop do?*
>Central Air = Y  →  [1, 0]
  Central Air = N  →  [0, 1]

This is the direct output of

```python
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    drop="first",              # drops one column per feature to avoid multicollinearity    sparse_output=False,       # returns dense numpy array instead of sparse matrix    handle_unknown="ignore"    # unseen categories at inference produce all-zero row)
encoded = ohe.fit_transform(df[["Central Air", "Street"]])
```

---
**3. Label Encoding**

Assigns an integer to each category without minding the order or value. `Blmngtn=0, Blueste=1, BrDale=2` and so on. Safe for tree-based models because trees find their own splits and do not interpret integer magnitude. Misleads linear models and neural networks.

In SKlearn.preprocessing there are two different encoders for that, where OridinalEncoder can be used in pipeline directly.

```python

# With sklearn -- useful inside ColumnTransformer pipelines (accepts 2D input)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["House Style"] = le.fit_transform(df["House Style"])
```

`LabelEncoder` accepts 1D input only and cannot be placed inside a `ColumnTransformer`. Use `OrdinalEncoder` without specifying categories when you need label encoding inside a pipeline.

```python
from sklearn.preprocessing import OrdinalEncoder

# OrdinalEncoder without categories argument behaves like LabelEncoder
# but accepts 2D input and works inside ColumnTransformer
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df[["House Style"]] = oe.fit_transform(df[["House Style"]])
```

---


**4. Target Encoding**

Replace each category with the mean of the target variable for that category. Produces a single numerical column regardless of cardinality, making it ideal for high-cardinality columns.

**Risk: data leakage.** If you compute target means on the full dataset before splitting, the encoding carries information from the validation and test sets into training. Always compute on the train set only.

```python
from sklearn.preprocessing import TargetEncoder

te = TargetEncoder(
    smooth="auto",   # automatically determines how much to pull toward global mean
    cv=5, # uses 5-fold cross-fitting internally to prevent leakage, AUTOMATICALLY, EVEN THOUGH NOT SET!!!!!!)
    target_type='continious' #if target is continious (like sales price), you might get an dimension explosion if this is not set!
X_train["Neighborhood_enc"] = te.fit_transform(X_train[["Neighborhood"]], y_train)
X_val["Neighborhood_enc"]   = te.transform(X_val[["Neighborhood"]])
```

  * Smooth and CV parameters here is implementation of **variations described in last note**

---


### Additional / Advanced Strategies

**5. Binary Encoding**

Assigns an integer to each category, converts that integer to its binary representation, and splits the bits into separate columns. A column with 25 categories produces 5 columns (`2^5 = 32 >= 25`) instead of 25.

```python
# pip install category_encoders
import category_encoders as ce

encoder = ce.BinaryEncoder(cols=["Neighborhood"])
X_train_enc = encoder.fit_transform(X_train)
X_val_enc   = encoder.transform(X_val)
```

The drawback is that the bit columns carry no semantic meaning. The model may detect spurious patterns across bit positions that do not reflect real relationships. Target encoding is usually preferred for high-cardinality columns.

---

**6. Frequency Encoding**

Replace each category with the proportion of times it appears in the dataset. Rare categories receive low values; common categories receive high values. Does not increase dimensionality and requires no target information.

```python
freq = X_train["Misc Feature"].value_counts(normalize=True)

X_train["Misc Feature_freq"] = X_train["Misc Feature"].map(freq)
X_val["Misc Feature_freq"]   = X_val["Misc Feature"].map(freq)

X_val["Misc Feature_freq"].fillna(0, inplace=True)   # unseen categories get 0
```

Useful for columns with many rare categories where the frequency of occurrence itself carries signal (rare feature = unusual house = potential price effect).

---

### Variations of Target Encoding

All three variants below address the same problem: standard target encoding overfits on rare categories and leaks target information. They differ in how they regularise the encoding.

**7. Smoothed Target Encoding**

Blends the category mean with the global mean, weighted by how many observations the category has.

**Categories with few observations are pulled toward the global mean; categories with many observations keep their own mean.  **

`m` is the smoothing parameter. A category with `n` observations receives weight `n / (n + m)` toward its own mean and `m / (n + m)` toward the global mean. With `m=10`, a category needs at least 10 observations before its own mean dominates.

`smooth = auto` lets sklearn calculate m statistically.
high variance -> less trustworthy -> higher m -> **more pull toward global mean**


  > global mean (average sales price of all houses) = 200.000
  > blue neighborhood: 100 houses, avg price = 150.000
  > green neighborhood: 3 houses, avg price: 300.000

Blue with 100 houses:
> Own weight: 100 / (100+10) = **0.91**
> Global weight: 10 / (100+10) = **0.09**
   Encoded = 0.91 × 150,000 + 0.09 × 200,000 = **154,500**
   Very close to its own mean, because there's enough data to trust it

Green with 3 houses:
> Own weight: 3 / (3+10) = **0.23**
> Global weight: 10 / (3+10) = **0.77**
> Encoded = 0.23 × 300,000 + 0.77 × 200,000 = **223,000**
> Pulled far from **300,000 toward the global mean**, because 3 observations are not reliable enough to trust

---

**8. K-Fold Target Encoding**

Splits the training set into K folds. Each row's encoding is computed from the folds that did not contain that row. Eliminates leakage entirely because a row never influences its own encoding.

#### Sklearn's `TargetEncoder` with `cv=5` does implement  K-Fold Target Encoding internally and is preferred for pipeline compatibility.

---

**9. CatBoost Encoding**

Processes rows in a random order. Each row's encoding is computed using only the rows that appeared before it in that order. No row influences its own encoding, which eliminates leakage without requiring cross-validation folds.

```python
import category_encoders as ce

encoder = ce.CatBoostEncoder(cols=["Neighborhood"], random_state=42)
X_train["Neighborhood_enc"] = encoder.fit_transform(X_train[["Neighborhood"]], y_train)
X_val["Neighborhood_enc"]   = encoder.transform(X_val[["Neighborhood"]])
```

CatBoost (the gradient boosting library) uses this encoding internally for all categorical columns, which is where the name comes from. It tends to perform well out of the box without requiring smoothing parameter tuning.

## Show Maximum Rows :
We have 82 columns here, but you might miss it as pycharm doesnt render "..." is not rendered with .show() function.

```
pd.set_option('display.max_rows', None) #setting
pd.reset_option('display.max_rows') # resetting
```

pandas' has a context manager where it could be set temporarily:

```
with pd.option_context('display.max_rows', None):
    display(df.isnull().sum())
```




In [2]:
#pd.set_option('display.max_rows', None)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   str    
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   str    
 7   Alley            198 non-null    str    
 8   Lot Shape        2930 non-null   str    
 9   Land Contour     2930 non-null   str    
 10  Utilities        2930 non-null   str    
 11  Lot Config       2930 non-null   str    
 12  Land Slope       2930 non-null   str    
 13  Neighborhood     2930 non-null   str    
 14  Condition 1      2930 non-null   str    
 15  Condition 2      2930 non-null   str    
 16  Bldg Type        2930 non-null   str    
 17  House Style      2930 non

In [5]:
df.head(30)

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900
5,6,527105030,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,6,2010,WD,Normal,195500
6,7,527127150,120,RL,41.0,4920,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,213500
7,8,527145080,120,RL,43.0,5005,Pave,NaN,IR1,HLS,...,0,NaN,NaN,NaN,0,1,2010,WD,Normal,191500
8,9,527146030,120,RL,39.0,5389,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,3,2010,WD,Normal,236500
9,10,527162130,60,RL,60.0,7500,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,6,2010,WD,Normal,189000


# Getting Lists of Strings and Cardinalities

In [6]:
df.select_dtypes(include="str") #gets the dataframe where the type of columns is just strings

categorical_cols = df.select_dtypes(include="str").columns #just the names of columns
print(len(categorical_cols))
categorical_cols

43


Index(['MS Zoning', 'Street', 'Alley', 'Lot Shape', 'Land Contour',
       'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
       'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC',
       'Central Air', 'Electrical', 'Kitchen Qual', 'Functional',
       'Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual',
       'Garage Cond', 'Paved Drive', 'Pool QC', 'Fence', 'Misc Feature',
       'Sale Type', 'Sale Condition'],
      dtype='str')

In [7]:
print(df[categorical_cols].nunique())

MS Zoning          7
Street             2
Alley              2
Lot Shape          4
Land Contour       4
Utilities          3
Lot Config         5
Land Slope         3
Neighborhood      28
Condition 1        9
Condition 2        8
Bldg Type          5
House Style        8
Roof Style         6
Roof Matl          8
Exterior 1st      16
Exterior 2nd      17
Mas Vnr Type       4
Exter Qual         4
Exter Cond         5
Foundation         6
Bsmt Qual          5
Bsmt Cond          5
Bsmt Exposure      4
BsmtFin Type 1     6
BsmtFin Type 2     6
Heating            6
Heating QC         5
Central Air        2
Electrical         5
Kitchen Qual       5
Functional         8
Fireplace Qu       5
Garage Type        6
Garage Finish      3
Garage Qual        5
Garage Cond        5
Paved Drive        3
Pool QC            4
Fence              4
Misc Feature       5
Sale Type         10
Sale Condition     6
dtype: int64


In [8]:
for col in categorical_cols:
    print(df[col].value_counts())
    print(f"[{col}]"+"*"*40)
    print( )

MS Zoning
RL         2273
RM          462
FV          139
RH           27
C (all)      25
I (all)       2
A (agr)       2
Name: count, dtype: int64
[MS Zoning]****************************************

Street
Pave    2918
Grvl      12
Name: count, dtype: int64
[Street]****************************************

Alley
Grvl    120
Pave     78
Name: count, dtype: int64
[Alley]****************************************

Lot Shape
Reg    1859
IR1     979
IR2      76
IR3      16
Name: count, dtype: int64
[Lot Shape]****************************************

Land Contour
Lvl    2633
HLS     120
Bnk     117
Low      60
Name: count, dtype: int64
[Land Contour]****************************************

Utilities
AllPub    2927
NoSewr       2
NoSeWa       1
Name: count, dtype: int64
[Utilities]****************************************

Lot Config
Inside     2140
Corner      511
CulDSac     180
FR2          85
FR3          14
Name: count, dtype: int64
[Lot Config]****************************************



# Handling Ordinal Columns

For `Columns Exter Qual/Cond, Bsmt Qual/Cond, Heating QC, Kitchen Qual, Garage Qual/Cond, Fireplace Qu, Pool QC`

we have repeating quality order like `Poor, Fair, Typical, Good,Excellent

For Paved Drive we have **Y, N and P** which might seem like One hot, but as p is semi paved, it can also be coded with 0 1 2.

### handle_unknown:

"error" →  throws an error (default)
"use_encoded_value" → uses the value set at "unknown value" (-1 here)`

insted of doing for each, we will use pipeline later.

In [9]:
from sklearn.preprocessing import OrdinalEncoder

quality_order = [["Po", "Fa", "TA", "Gd", "Ex"]]

encoder = OrdinalEncoder(
    categories=quality_order,
    handle_unknown="use_encoded_value", #to be ab
    unknown_value=-1        # unseen categories at inference get -1
)

df["Exter Qual_enc"] = encoder.fit_transform(df[["Exter Qual"]])

In [10]:
columns = ["Exter Qual_enc","Exter Qual"]
df[columns].head(10)

,Exter Qual_enc,Exter Qual
0,2.0,TA
1,2.0,TA
2,2.0,TA
3,3.0,Gd
4,2.0,TA
5,2.0,TA
6,3.0,Gd
7,3.0,Gd
8,3.0,Gd
9,2.0,TA


In [11]:
pavement_order = [["N","P","Y"]]

encoder = OrdinalEncoder(
    categories=pavement_order,
    handle_unknown="use_encoded_value", #to be ab
    unknown_value=-1       # unseen categories at inference get -1
)

df["Paved Drive_enc"] = encoder.fit_transform(df[["Paved Drive"]])


# One Hot Encoding
For example for

* Central air with yes and no,
* Street with pave and gravel

In [12]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    drop="first",              # drops one column per feature to avoid multicollinearity
    sparse_output=False,       # returns dense numpy array instead of sparse matrix
    handle_unknown="ignore"    # unseen categories at inference produce all-zero row
)

ohe.set_output(transform="pandas") # to recieve pandas data with column names

encoded = ohe.fit_transform(df[["Central Air", "Street"]])
display(encoded.head(10))

df = pd.concat([df,encoded],axis=1)


,Central Air_Y,Street_Pave
0,1.0,1.0
1,1.0,1.0
2,1.0,1.0
3,1.0,1.0
4,1.0,1.0
5,1.0,1.0
6,1.0,1.0
7,1.0,1.0
8,1.0,1.0
9,1.0,1.0


# Label Encoding with Ordinal Encoder

In [13]:
df["Neighborhood"].unique()

<ArrowStringArray>
[  'NAmes', 'Gilbert', 'StoneBr',  'NWAmes', 'Somerst',  'BrDale', 'NPkVill',
 'NridgHt', 'Blmngtn', 'NoRidge', 'SawyerW',  'Sawyer',  'Greens', 'BrkSide',
 'OldTown',  'IDOTRR', 'ClearCr',   'SWISU', 'Edwards', 'CollgCr', 'Crawfor',
 'Blueste', 'Mitchel',  'Timber', 'MeadowV', 'Veenker', 'GrnHill', 'Landmrk']
Length: 28, dtype: str

In [14]:

from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
oe.set_output(transform="pandas")
labels_generated = oe.fit_transform(df[["Neighborhood"]])

In [15]:
pd.concat([df[["Neighborhood"]], labels_generated], axis=1)

,Neighborhood,Neighborhood
0,NAmes,15.0
1,NAmes,15.0
2,NAmes,15.0
3,NAmes,15.0
4,Gilbert,8.0
...,...,...
2925,Mitchel,14.0
2926,Mitchel,14.0
2927,Mitchel,14.0
2928,Mitchel,14.0


# Target Encoding

In [31]:
from sklearn.preprocessing import TargetEncoder


X_neighborhood = df[["Neighborhood"]]
y = df["SalePrice"]

X_val, y_train, y_val = train_test_split(X_neighborhood, y, test_size=0.2, random_state=42) # As we SHOULD NOT LEAK DATA FROM TEST SET, WE WILL TRAIN ONCE and then just transform

te = TargetEncoder(cv= 5,smooth="auto", target_type="continuous")
te.set_output(transform="pandas")

print(df["Neighborhood"].value_counts())
print(df["Neighborhood"].nunique())
print(X_train.value_counts() )

neighborhood_train_encoded = te.fit_transform(X_train, y_train) #WE TRAINED/FITTED WITH X_Train and y_Train data
neighborhood_valuation_encoded   = te.transform(X_val) # WE JUST HAVE TRANSFORMED the X_VAL, NOT TRAINED AGAIN.



Neighborhood
NAmes      443
CollgCr    267
OldTown    239
Edwards    194
Somerst    182
NridgHt    166
Gilbert    165
Sawyer     151
NWAmes     131
SawyerW    125
Mitchel    114
BrkSide    108
Crawfor    103
IDOTRR      93
Timber      72
NoRidge     71
StoneBr     51
SWISU       48
ClearCr     44
MeadowV     37
BrDale      30
Blmngtn     28
Veenker     24
NPkVill     23
Blueste     10
Greens       8
GrnHill      2
Landmrk      1
Name: count, dtype: int64
28
Neighborhood
NAmes           367
CollgCr         211
OldTown         199
Edwards         150
Gilbert         138
Somerst         136
NridgHt         121
Sawyer          119
NWAmes          109
SawyerW          98
Mitchel          97
Crawfor          89
BrkSide          84
IDOTRR           73
Timber           59
NoRidge          57
SWISU            37
StoneBr          36
ClearCr          35
MeadowV          28
BrDale           24
Blmngtn          23
Veenker          22
NPkVill          15
Blueste           7
Greens            7
GrnHi

In [32]:
neighborhood_train_encoded

,Neighborhood
381,188720.741152
834,219567.139879
1898,146754.165182
678,146754.165182
700,124143.141097
...,...
1638,191718.856255
1095,191530.602437
1130,221411.964937
1294,123192.267203
